Preprocessing

In [4]:
import os
import cv2

# Parameters
image_folder = "seamounts_bboxes"  # Root directory of the dataset
output_folder = "seamounts_bboxes_cropped"  # Temporary folder for cropped images
# image_size = (128, 128)  # Image dimensions for the model
crop_pixels = 65  # Pixels to crop from each border

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Function to crop borders from an image
def crop_fixed_border(image, crop_pixels=5):
    height, width, _ = image.shape
    if height > crop_pixels * 2 and width > crop_pixels * 2:
        cropped_image = image[crop_pixels:height-crop_pixels, crop_pixels:width-crop_pixels]
        return cropped_image
    else:
        return image  # Return original image if cropping not possible

# Preprocess images (cropping and resizing)
print("Preprocessing images...")

# Loop through all files in the image folder
for filename in os.listdir(image_folder):
    image_path = os.path.join(image_folder, filename)
    
    # Check if the file is an image (you can expand the check for other image formats)
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        # Read the image
        image = cv2.imread(image_path)
        
        # Crop the image
        cropped_image = crop_fixed_border(image, crop_pixels)
        
        # Save the cropped image to the output folder
        output_image_path = os.path.join(output_folder, filename)
        cv2.imwrite(output_image_path, cropped_image)

print("Preprocessing complete.")


Preprocessing images...
Preprocessing complete.


Matching

In [14]:
import cv2
import os
import numpy as np
import pandas as pd

# Function to find smaller map tile in the corresponding larger map tile using ORB feature matching and homography
def find_and_mark_tiles_with_homography(large_tile_folder, small_tile_folder, output_csv):
    # List to store results
    results = []
    
    # Get list of files in both folders
    large_tiles = os.listdir(large_tile_folder)
    small_tiles = os.listdir(small_tile_folder)

    # Debug: print the list of files in both folders
    print("Large tiles:", large_tiles)
    print("Small tiles:", small_tiles)

    # Create a dictionary of small tiles indexed by PEAKID
    small_tiles_dict = {}
    for small_tile_name in small_tiles:
        # Extract the PEAKID from the smaller tile's filename (assuming format [PEAKID]_1.png)
        peak_id = small_tile_name.split('_')[0]
        small_tiles_dict[peak_id] = small_tile_name

    # Debug: print the small tiles dictionary
    print("Small tiles dictionary:", small_tiles_dict)

    # Initialize ORB detector
    orb = cv2.ORB_create()

    # Iterate through each large tile
    for large_tile_name in large_tiles:
        # Extract the PEAKID from the larger tile's filename (assuming format [PEAKID].png)
        peak_id = large_tile_name.split('.')[0]

        # Ensure the PEAKID matches the format of the small tile (i.e., keep the '.0' if present)
        peak_id = str(float(peak_id))  # This will preserve the ".0" if present
        
        # Debug: print the current large tile being processed
        print(f"Processing large tile: {large_tile_name}, PEAKID: {peak_id}")
        
        # Check if there is a corresponding small tile for this PEAKID
        if peak_id in small_tiles_dict:
            small_tile_name = small_tiles_dict[peak_id]
            large_tile_path = os.path.join(large_tile_folder, large_tile_name)
            small_tile_path = os.path.join(small_tile_folder, small_tile_name)

            # Debug: print the paths of the current large and small tiles
            print(f"Found corresponding small tile: {small_tile_name}")
            print(f"Large tile path: {large_tile_path}")
            print(f"Small tile path: {small_tile_path}")

            # Read the images
            large_tile = cv2.imread(large_tile_path, cv2.IMREAD_COLOR)
            small_tile = cv2.imread(small_tile_path, cv2.IMREAD_COLOR)

            if large_tile is None:
                print(f"Error: Unable to load large tile {large_tile_path}")
                continue
            if small_tile is None:
                print(f"Error: Unable to load small tile {small_tile_path}")
                continue

            # Convert to grayscale (ORB works in grayscale)
            gray_large_tile = cv2.cvtColor(large_tile, cv2.COLOR_BGR2GRAY)
            gray_small_tile = cv2.cvtColor(small_tile, cv2.COLOR_BGR2GRAY)

            # Detect keypoints and descriptors using ORB
            kp_large, des_large = orb.detectAndCompute(gray_large_tile, None)
            kp_small, des_small = orb.detectAndCompute(gray_small_tile, None)

            # Ensure descriptors are not None
            if des_large is None or des_small is None:
                print(f"Error: No descriptors found for large or small tile.")
                continue

            # Use BFMatcher to match descriptors
            bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
            matches = bf.match(des_large, des_small)

            # Sort matches by distance (best matches first)
            matches = sorted(matches, key=lambda x: x.distance)

            # Debug: print the number of matches
            print(f"Number of matches found: {len(matches)}")

            # If we have enough good matches (e.g., more than 10)
            if len(matches) > 10:
                # Extract the matching keypoints
                pts_large = []
                pts_small = []
                for match in matches[:10]:  # Consider only the best 10 matches
                    pts_large.append(kp_large[match.queryIdx].pt)
                    pts_small.append(kp_small[match.trainIdx].pt)

                # Convert to np.float32
                pts_large = np.float32(pts_large)
                pts_small = np.float32(pts_small)

                # Compute the homography matrix
                homography, mask = cv2.findHomography(pts_small, pts_large, cv2.RANSAC, 5.0)

                # Get the corners of the small tile
                h, w = small_tile.shape[:2]
                corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]]).reshape(-1, 1, 2)

                # Transform the corners to the large tile's coordinate system
                transformed_corners = cv2.perspectiveTransform(corners, homography)

                # Draw the bounding box around the matched region
                large_tile_with_box = large_tile.copy()
                cv2.polylines(large_tile_with_box, [np.int32(transformed_corners)], isClosed=True, color=(0, 255, 0), thickness=3)

                # Store the coordinates of the bounding box
                top_left = tuple(np.int32(transformed_corners[0][0]))
                bottom_right = tuple(np.int32(transformed_corners[2][0]))

                results.append({
                    'large_tile': large_tile_name,
                    'small_tile': small_tile_name,
                    'top_left_x': top_left[0],
                    'top_left_y': top_left[1],
                    'bottom_right_x': bottom_right[0],
                    'bottom_right_y': bottom_right[1]
                })

                # Optionally, save the marked image to a file
                marked_image_path = os.path.join('marked_images', f"marked_{large_tile_name}")
                os.makedirs('marked_images', exist_ok=True)
                cv2.imwrite(marked_image_path, large_tile_with_box)
                print(f"Marked image saved to: {marked_image_path}")

            else:
                print(f"Not enough good matches found for {large_tile_name} and {small_tile_name}")

        else:
            print(f"No corresponding small tile found for PEAKID: {peak_id}")

    # Debug: print the results
    print(f"Found {len(results)} matches.")
    
    # Convert the results to a pandas DataFrame and save to CSV
    if results:
        df = pd.DataFrame(results)
        df.to_csv(output_csv, index=False)
        print(f"Results saved to {output_csv}")
    else:
        print("No matches to save.")

# Example usage
find_and_mark_tiles_with_homography('seamounts_galore_cropped', 'seamounts_bboxes_cropped', 'output_pixel_coordinates.csv')


Large tiles: ['1008448.0.png', '1011009.0.png', '1011816.0.png', '1013921.0.png', '1018681.0.png', '1025041.0.png', '1025051.0.png', '1025969.0.png', '1028494.0.png', '1029117.0.png', '1032464.0.png', '1033749.0.png', '1045988.0.png', '1046854.0.png', '1050802.0.png', '1062546.0.png', '1133273.0.png', '1134465.0.png', '1134466.0.png', '1137997.0.png', '1139232.0.png', '1140976.0.png', '1141217.0.png', '1141528.0.png', '1143763.0.png', '1186819.0.png', '1208725.0.png', '1214694.0.png', '1258577.0.png', '1265410.0.png', '1281864.0.png', '1293729.0.png', '1296008.0.png', '1318921.0.png', '1752990.0.png', '2076953.0.png', '2079834.0.png', '2091324.0.png', '2093999.0.png', '2112638.0.png', '2146655.0.png', '2169437.0.png', '2196645.0.png', '2205479.0.png', '2238745.0.png', '2252312.0.png', '2254185.0.png', '2276053.0.png', '2276072.0.png', '2278391.0.png', '2284600.0.png', '2296206.0.png', '2314845.0.png', '2334408.0.png', '2335718.0.png', '2341351.0.png', '2341721.0.png', '2355160.0.png', 